# Anime Stitch Pipeline (ASP) — Stage-by-Stage Walkthrough

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ACFPeacekeeper/Image-Toolkit/main?filepath=docs/notebooks/asp_pipeline_walkthrough.ipynb)

This notebook executes the full 13-stage ASP on a 5-frame test sequence and
visualises each stage's output, so you can understand what the pipeline does
without reading 4,000 lines of `pipeline.py`.

**Prerequisites**:
```bash
source .venv/bin/activate
pip install matplotlib scikit-image
```

**GPU**: BiRefNet (Stage 4) and feature matching (Stage 5) are GPU-optional.
Both fall back to CPU automatically. Mark GPU cells with `# SKIP_CI` to exclude
them from `nbconvert --execute` CI runs.

In [ ]:
import sys
import pathlib
import glob
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Add repo root to sys.path so backend imports work
REPO_ROOT = pathlib.Path().resolve().parents[1]  # docs/notebooks/ → repo root
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# ── Dataset selection ──────────────────────────────────────────────────────────
TEST_ID = 1   # Change this to any test 1–97

DATA_DIR = REPO_ROOT / 'data' / f'asp_test{TEST_ID}'
OUT_DIR  = DATA_DIR / 'output' / 'walkthrough'
OUT_DIR.mkdir(parents=True, exist_ok=True)

image_paths = sorted(glob.glob(str(DATA_DIR / '*.png')) + glob.glob(str(DATA_DIR / '*.jpg')))
print(f'Dataset: {DATA_DIR}')
print(f'Frames found: {len(image_paths)}')
for p in image_paths:
    print(f'  {pathlib.Path(p).name}')

## Stage 0 — Source frames

Load and display the raw source frames before any processing.

In [ ]:
frames_bgr = [cv2.imread(p) for p in image_paths]
frames_rgb = [cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in frames_bgr]

N = len(frames_rgb)
fig, axes = plt.subplots(1, N, figsize=(4 * N, 5))
if N == 1:
    axes = [axes]
for i, (ax, img) in enumerate(zip(axes, frames_rgb)):
    ax.imshow(img)
    ax.set_title(f'Frame {i} ({img.shape[1]}×{img.shape[0]})')
    ax.axis('off')
fig.suptitle(f'Stage 0 — Source frames (asp_test{TEST_ID})', fontsize=14)
plt.tight_layout()
plt.show()

## Stage 1–2 — Frame selection + width normalisation

Smart frame selection drops hold frames (duplicate poses) and near-duplicates.
Width normalisation Lanczos-resamples all frames to the same width.

In [ ]:
from backend.src.animation.frame_selection import smart_select_frames

thumbs = [cv2.resize(f, (128, int(128 * f.shape[0] / f.shape[1]))) for f in frames_bgr]
selected_paths, selected_thumbs = smart_select_frames(
    frames_paths=image_paths,
    thumbs=thumbs,
)

print(f'Smart-selected {len(selected_paths)} / {len(image_paths)} frames')
dropped = [p for p in image_paths if p not in selected_paths]
if dropped:
    print(f'Dropped: {[pathlib.Path(p).name for p in dropped]}')

## Run the full pipeline

`AnimeStitchPipeline.run()` executes all 13 stages and saves intermediate
outputs to `data/asp_testN/output/panorama_stages/`. Inspect those files
for the intermediate visualisations (Stage 2 normalised frames, Stage 4 masks,
Stage 9 temporal render, Stage 11 composite).

In [ ]:
# SKIP_CI  ← GPU-dependent cell; excluded from nbconvert --execute
from backend.src.animation.pipeline import AnimeStitchPipeline

output_path = str(OUT_DIR / 'panorama.png')
pipeline = AnimeStitchPipeline(debug=True)  # debug=True writes panorama_stages/

result = pipeline.run(
    image_paths=image_paths,
    output_path=output_path,
)
print(f'Pipeline complete → {output_path}')
print(f'Canvas size: {result.width}×{result.height}')

## Stage 9 — Temporal render (background canvas)

The median render of all warped frames — before foreground compositing.

In [ ]:
# SKIP_CI
stage9_path = DATA_DIR / 'output' / 'panorama_stages' / 'stage09_temporal_render.png'
final_path  = pathlib.Path(output_path)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, path, title in zip(axes,
                            [stage9_path, final_path],
                            ['Stage 9 — Temporal render', 'Stage 13 — Final composite']):
    if path.exists():
        img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f'{title} ({img.shape[1]}×{img.shape[0]})')
    else:
        ax.set_title(f'{title} — not found')
    ax.axis('off')
fig.suptitle('Temporal render vs final composite', fontsize=14)
plt.tight_layout()
plt.show()

## Alignment visualisation — translation vectors

The bundle-adjusted affines are loaded from Stage 8's JSON log.
A uniform staircase indicates a clean vertical pan; a flat or reversed
segment indicates an alignment failure.

In [ ]:
# SKIP_CI
import json

info_path = DATA_DIR / 'output' / 'panorama_stages' / 'stage08_canvas_info.json'
if info_path.exists():
    info = json.loads(info_path.read_text())
    affines = info.get('affines', [])
    tx = [a.get('tx', 0) for a in affines]
    ty = [a.get('ty', 0) for a in affines]
    frames_idx = list(range(len(affines)))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(frames_idx, tx, marker='o', color='steelblue')
    ax1.set_title('tx (horizontal shift) per frame')
    ax1.set_xlabel('Frame index')
    ax1.set_ylabel('px')
    ax2.plot(frames_idx, ty, marker='o', color='tomato')
    ax2.set_title('ty (vertical shift) per frame')
    ax2.set_xlabel('Frame index')
    ax2.set_ylabel('px')
    plt.suptitle('Bundle-adjusted translation vectors', fontsize=13)
    plt.tight_layout()
    plt.show()

    print('Alignment health:', info.get('health', 'not recorded'))
else:
    print(f'stage08_canvas_info.json not found at {info_path}')

## Seam quality — gradient heatmap

High-energy bands at horizontal seam rows indicate abrupt brightness transitions.
A well-blended stitch should have diffuse, low-energy heatmap values.

In [ ]:
# SKIP_CI
if final_path.exists():
    final_bgr = cv2.imread(str(final_path))
    gray = cv2.cvtColor(final_bgr, cv2.COLOR_BGR2GRAY).astype(float)
    sobel = np.abs(cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    ax1.imshow(cv2.cvtColor(final_bgr, cv2.COLOR_BGR2RGB))
    ax1.set_title('Final output')
    ax1.axis('off')
    im = ax2.imshow(sobel, cmap='inferno', vmax=np.percentile(sobel, 99))
    fig.colorbar(im, ax=ax2, shrink=0.7)
    ax2.set_title('Gradient magnitude (Sobel-Y) — seam heatmap')
    ax2.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f'Output not found at {final_path} — run the pipeline cell first.')